# Navier–Stokes (Kolmogorov Re=500): **AWFNO-H10-K10** — competitive history-conditioned study

**Purpose:** information-matched alternative to U-NO-H10. Observe exactly 10 truth frames (`t0..t9`) once, predict the next 10 frames using the **original alternating local–Fourier–local AWFNO**, and autoregressively roll forward to `t100` without using true future frames.

- Model: same four-layer AWFNO backbone as the original repository, `WIDTH=32`, with its input projection widened to 10 historical frames; no additional architecture changes.
- Training: 10 observed frames → 10 future frames, train-only scalar normalization and independent validation selection on a **forecast-only physical L2 error**.
- Metric: physical-space absolute `L2` integrated over **frames 10..100**, and final-time spatial absolute `L2`.
- Dataset split: **200/30/70 exact file indices**. Training sizes: **[25,50,100,200]**.
- GPU training has **not** been executed as part of preparing this notebook; results will appear only after execution.
- A default 70-epoch budget mirrors the recent U-NO-H10 studies, but architectures, loss objectives, windows and optimizer steps differ, so this is **not a compute-matched comparison**.
- Original AWFNO source is pinned to commit `0c7b39611f2ebc8b868561a376b7fd0211ea7ddc`. At runtime, Colab retrieves it directly; no `.py` upload is needed.


In [ ]:

# Run in Google Colab with a GPU runtime.
from google.colab import drive
drive.mount('/content/drive')
import os, json, hashlib, random, time, gc, math, copy, urllib.request, importlib.util, csv
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), 'Select a GPU runtime in Colab before training.'
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)


In [ ]:

CFG = {
    'PDE': 'navier_stokes',
    'DATA_PATH': '/content/drive/MyDrive/Kolmogorov_NS_Re500_start0.25_horizon0.25_dt0.0025_v1/kolmogorov_dataset.pt',
    'OUTPUT_ROOT': '/content/drive/MyDrive/AWFNO_NS_Re500_AWFNO_H10_K10_JCP',
    'N_SIZES': [25,50,100,200],
    'HISTORY': 10,
    'WINDOW_K': 10,
    'WINDOW_STRIDE': 10,
    'EPOCHS': 70,                 # Same epoch budget as the recent U-NO-H10 studies, not identical compute.
    'BATCH_SIZE': 4,
    'EVAL_BATCH_SIZE': 2,
    'LR': 1e-3,
    'WEIGHT_DECAY': 1e-6,
    'GRAD_CLIP': 1.0,
    'VAL_EVERY': 5,
    'PATIENCE_EVALS': 12,
    'SEED': 1234,
    'FORCE_RETRAIN': False,
    'VERIFY_NS_FILE_SHA256': False, # Metadata hash checked; optional expensive full-dataset hash.
}
assert CFG['HISTORY']==10 and CFG['WINDOW_K']==10 and CFG['WINDOW_STRIDE']==10
ROOT=Path(CFG['OUTPUT_ROOT']); ROOT.mkdir(parents=True,exist_ok=True)
print(json.dumps(CFG,indent=2))


In [ ]:

# Import the *actual published AWFNO layers* from your GitHub repo at a pinned commit.
# No separate .py upload is required. The repository file is downloaded in this cell.
SOURCE_COMMIT = '0c7b39611f2ebc8b868561a376b7fd0211ea7ddc'
SOURCE_SUBPATH = 'experiments/navier_stokes_2d/awfno.py'
SOURCE_URL = f'https://raw.githubusercontent.com/dimigourz/AWFNO/{SOURCE_COMMIT}/{SOURCE_SUBPATH}'
local_source = Path('/content/awfno_original_h10_import.py')
urllib.request.urlretrieve(SOURCE_URL,local_source)
source_hash = hashlib.sha256(local_source.read_bytes()).hexdigest()
print('Original AWFNO source:',SOURCE_URL,'SHA256:',source_hash)

# The original module sets its paths and hyperparameters at import time.
os.environ.update({
    'SEED':str(CFG['SEED']),
    'DATA_PATH':str(CFG['DATA_PATH']),
    'OUTPUT_ROOT':str(ROOT/'original_source_import_cache'),
    'NS_DATA_DIR':str(Path(CFG['DATA_PATH']).parent),
    'NS_OUTPUT_ROOT':str(ROOT/'original_source_import_cache'),
    'NS_SIZES':','.join(map(str,CFG['N_SIZES'])),
    'WINDOW_K':str(CFG['WINDOW_K']),
    'WINDOW_STRIDE':str(CFG['WINDOW_STRIDE']),
    'NS_K':str(CFG['WINDOW_K']),
    'NS_TB_RANK':str(CFG['WINDOW_K']), # Original source requires rank <= K.
    'NS_STRIDE':str(CFG['WINDOW_STRIDE']),
})
spec=importlib.util.spec_from_file_location('awfno_original_h10_import',local_source)
base=importlib.util.module_from_spec(spec)
spec.loader.exec_module(base)


In [ ]:

# Verified Kolmogorov Re=500, 101-frame benchmark; exact file-defined split 200/30/70.
path=Path(CFG['DATA_PATH']); assert path.is_file(),f'Dataset missing: {path}'
meta_file=path.parent/'metadata.json'
assert meta_file.is_file(), 'Expected verified metadata.json adjacent to the dataset.'
meta=json.loads(meta_file.read_text())
EXPECTED_SHA='7030bc43e6331cdf472182ae0ed4f2684c18da8d336ca9ed92c37baf0bfc8a06'
assert meta['status']=='PASS' and meta['sha256']==EXPECTED_SHA, 'Metadata does not match verified dataset.'
assert meta['shape']==[300,128,128,101]
if CFG['VERIFY_NS_FILE_SHA256']:
    hasher=hashlib.sha256()
    with path.open('rb') as fh:
        for block in iter(lambda:fh.read(8*1024*1024),b''):
            hasher.update(block)
    assert hasher.hexdigest()==EXPECTED_SHA, 'Dataset SHA mismatch'
raw=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
assert raw['U'].shape==(300,128,128,101)
assert torch.equal(raw['train_indices'].cpu(),torch.arange(200))
assert torch.equal(raw['val_indices'].cpu(),torch.arange(200,230))
assert torch.equal(raw['test_indices'].cpu(),torch.arange(230,300))
U=raw['U'].numpy()  # mmap backed; avoid copying 300 trajectories
TRAIN,VAL,TEST=U[:200],U[200:230],U[230:300]
x=np.asarray(raw['x'],dtype=np.float32).reshape(-1)
y=np.asarray(raw['y'],dtype=np.float32).reshape(-1)
TIMES=np.asarray(raw['times'],dtype=np.float64).reshape(-1)
assert U.shape==(300,128,128,101)
assert np.allclose(np.diff(TIMES),0.0025,atol=2e-7)
base.set_geometry((300,128,128,101,1),x,y,TIMES)
WEIGHTS=np.full((128,128),float(base.DX*base.DY),dtype=np.float64)
DT=float(np.mean(np.diff(TIMES)))
print('Kolmogorov data:',U.shape,'train/val/test:',len(TRAIN),len(VAL),len(TEST))


In [ ]:

# Original AWFNO encoder, unchanged Fourier/local blocks. Only input projection now accepts H=10 states.
class AWFNOH10(nn.Module):
    def __init__(self):
        super().__init__()
        self.core=base.DirectWindowModel(c_dyn=1,K=CFG['WINDOW_K'],encoder_kind='AW')
        self.core.encoder=base.AWEncoder(CFG['HISTORY'])
    def forward(self,history_normalized): # (B,128,128,10) -> (B,128,128,10)
        assert history_normalized.ndim==4 and history_normalized.shape[-1]==10
        out=self.core(history_normalized)[...,0]
        return out


In [ ]:

H=CFG['HISTORY'];K=CFG['WINDOW_K'];NT=U.shape[-1]
assert NT==101 and H==K==10
assert max(CFG['N_SIZES'])<=len(TRAIN)
assert len(VAL)==30 and len(TEST)==70

def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def train_stats(U_train):
    # Train-only mean/std. Bounded temporary memory (stream over trajectories).
    s=sq=count=0.0
    for i in range(len(U_train)):
        a=np.asarray(U_train[i],dtype=np.float64)
        s+=float(a.sum());sq+=float(np.square(a).sum());count+=a.size
    mean=s/count; std=math.sqrt(max(sq/count-mean**2,1.e-12))
    return float(mean),float(std)

class HistoryWindows(Dataset):
    def __init__(self,trajectories,stride=10):
        self.U=trajectories
        last=self.U.shape[-1]-H-K
        self.starts=list(range(0,last+1,stride))
        if self.starts[-1]!=last:self.starts.append(last) # include final t100 in training
        assert self.starts and max(self.starts)+H+K<=NT
    def __len__(self):return len(self.U)*len(self.starts)
    def __getitem__(self,index):
        sid=index//len(self.starts);start=self.starts[index%len(self.starts)]
        hist=np.array(self.U[sid,...,start:start+H],copy=True)
        future=np.array(self.U[sid,...,start+H:start+H+K],copy=True)
        return torch.from_numpy(hist),torch.from_numpy(future)

@torch.no_grad()
def forecast(model,trajectories,mean,std,batch_size=None):
    # Only t0..t9 are observed, never feed true t>=10 into the model.
    model.eval();bs=batch_size or CFG['EVAL_BATCH_SIZE']
    pred=np.empty_like(trajectories,dtype=np.float32)
    pred[...,:H]=trajectories[...,:H]
    for off in range(0,len(trajectories),bs):
        last=min(off+bs,len(trajectories))
        hist=torch.as_tensor(np.array(trajectories[off:last,...,:H],copy=True),device=DEVICE)
        for pos in range(H,NT,K):
            next_norm=model((hist-mean)/std)
            assert next_norm.shape==(*hist.shape[:-1],K)
            future=next_norm*std+mean
            take=min(K,NT-pos)
            pred[off:last,...,pos:pos+take]=future[...,:take].cpu().numpy()
            if not torch.isfinite(future).all():
                raise FloatingPointError(f'Nonfinite prediction at index {pos}')
            if pos+take<NT:
                hist=torch.cat((hist,future[...,:take]),dim=-1)[...,-H:]
    return pred

def metrics(pred,true):
    assert pred.shape==true.shape and pred.shape[-1]==NT
    e2=np.empty((len(pred),NT-H),dtype=np.float64)
    for i in range(0,len(pred),2):
        err=np.asarray(pred[i:i+2,...,H:]-true[i:i+2,...,H:],dtype=np.float64)
        e2[i:i+2]=np.einsum('bxyk,xy->bk',err*err,WEIGHTS,optimize=True)
    wt=np.ones(NT-H,dtype=np.float64);wt[0]=wt[-1]=0.5
    full=np.sqrt(np.sum(e2*wt[None,:],axis=1)*DT)
    final=np.sqrt(e2[:,-1])
    first=np.sqrt(e2[:,0])
    return {
       'forecast_l2_mean':float(full.mean()),'forecast_l2_sd':float(full.std()),
       'final_l2_mean':float(final.mean()),'final_l2_sd':float(final.std()),
       'first_forecast_l2_mean':float(first.mean()),
       'l2_vs_time_mean':np.sqrt(e2).mean(axis=0).tolist(),
       'n_test':int(len(true)), 'forecast_start_frame':H,'forecast_end_frame':NT-1,
    }

# Simple unit check for the physical integration and forecast-only masking.
zero=np.zeros((1,4,5,NT),dtype=np.float32)
assert HistoryWindows(zero).starts[-1]==81
assert HistoryWindows(zero)[0][0].shape==(4,5,H)
assert HistoryWindows(zero)[0][1].shape==(4,5,K)
assert np.allclose(metrics(np.zeros((1,*WEIGHTS.shape,NT),np.float32),
                           np.zeros((1,*WEIGHTS.shape,NT),np.float32))['forecast_l2_mean'],0)

seed_everything(CFG['SEED'])
smoke=AWFNOH10().to(DEVICE)
xsm=torch.randn(1,*WEIGHTS.shape,H,device=DEVICE)
ysm=smoke(xsm)
assert ysm.shape==(1,*WEIGHTS.shape,K),ysm.shape
assert torch.isfinite(ysm).all()
ysm.square().mean().backward()
assert any(p.grad is not None for p in smoke.parameters())
PARAMS=sum(p.numel() for p in smoke.parameters() if p.requires_grad)
REAL_DOF=sum(p.numel()*(2 if p.is_complex() else 1) for p in smoke.parameters() if p.requires_grad)
print('AWFNO-H10 smoke passed: output',tuple(ysm.shape),'parameters',PARAMS,'real DOF',REAL_DOF)
del smoke,xsm,ysm
torch.cuda.empty_cache()


In [ ]:

# Model selection: validation FORECAST-only physical L2 over t10..t100.
# Outputs: per-N checkpoints, JSON and a summary.csv. Never evaluates on test before selection.
def signature_for(n):
    d={k:v for k,v in CFG.items() if k not in ('FORCE_RETRAIN','VERIFY_NS_FILE_SHA256')}
    d.update({'n_train':n,'source_commit':SOURCE_COMMIT,'source_hash':source_hash,
              'data_expected_sha':(EXPECTED_SHA if CFG['PDE']=='navier_stokes' else None)})
    return hashlib.sha256(json.dumps(d,sort_keys=True).encode()).hexdigest()[:14]

def train_one(n):
    seed_everything(CFG['SEED']+n)
    out=ROOT/f'N{n}';out.mkdir(parents=True,exist_ok=True)
    signature=signature_for(n)
    result_path=out/'result.json'
    if result_path.is_file() and not CFG['FORCE_RETRAIN']:
        previous=json.loads(result_path.read_text())
        if previous['signature']!=signature:
            raise RuntimeError(f'Configuration changed for N={n}; use a new OUTPUT_ROOT.')
        print('Reusing verified completed experiment:',result_path)
        return previous

    mean,std=train_stats(TRAIN[:n]);assert std>0
    tr_ds=HistoryWindows(TRAIN[:n],stride=CFG['WINDOW_STRIDE'])
    print(f'N={n} windows/trajectory={len(tr_ds.starts)}, normalization={mean:+.5e}/{std:.5e}')
    gen=torch.Generator().manual_seed(CFG['SEED']+n)
    loader=DataLoader(tr_ds,batch_size=CFG['BATCH_SIZE'],shuffle=True,
                      generator=gen,num_workers=0,pin_memory=True)
    model=AWFNOH10().to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=CFG['LR'],weight_decay=CFG['WEIGHT_DECAY'])
    best_val=float('inf');best_epoch=0;since_best=0;elapsed=0;start_epoch=1;history=[]
    best_ckpt=out/'best.pt';latest_ckpt=out/'latest.pt'
    if latest_ckpt.is_file() and not CFG['FORCE_RETRAIN']:
        ckpt=torch.load(latest_ckpt,map_location='cpu',weights_only=False)
        if ckpt['signature']!=signature:raise RuntimeError('Checkpoint config mismatch')
        model.load_state_dict(ckpt['model']);optimizer.load_state_dict(ckpt['optimizer'])
        best_val=ckpt['best_val'];best_epoch=ckpt['best_epoch'];since_best=ckpt['since_best']
        elapsed=ckpt['elapsed'];history=ckpt['history'];start_epoch=ckpt['epoch']+1
        gen.set_state(ckpt['loader_rng'])
        torch.set_rng_state(ckpt['torch_rng'])
        if ckpt.get('cuda_rng') is not None:torch.cuda.set_rng_state_all(ckpt['cuda_rng'])
        random.setstate(ckpt['py_rng']);np.random.set_state(ckpt['numpy_rng'])
        print('Resuming N',n,'from epoch',start_epoch)
    run_start=time.perf_counter()
    for epoch in range(start_epoch,CFG['EPOCHS']+1):
        t0=time.perf_counter();model.train();loss_sum=0;n_batches=0
        for inp,target in loader:
            inp=inp.to(DEVICE,non_blocking=True);target=target.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred_norm=model((inp-mean)/std)
            loss=F.mse_loss(pred_norm,(target-mean)/std)
            if not torch.isfinite(loss):raise FloatingPointError(f'Nonfinite loss, N={n} epoch={epoch}')
            loss.backward()
            grad=torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['GRAD_CLIP'])
            if not torch.isfinite(grad):raise FloatingPointError('Nonfinite gradient')
            optimizer.step()
            loss_sum+=float(loss.detach());n_batches+=1
        # Checkpoint data written only after all prior CUDA work is synchronized.
        torch.cuda.synchronize();epoch_sec=time.perf_counter()-t0
        row={'epoch':epoch,'train_normalized_mse':loss_sum/max(n_batches,1),
             'epoch_seconds':epoch_sec}
        if epoch==1 or epoch%CFG['VAL_EVERY']==0 or epoch==CFG['EPOCHS']:
            val_pred=forecast(model,VAL,mean,std)
            val=metrics(val_pred,VAL)['forecast_l2_mean']
            row['val_forecast_l2']=val
            if val<best_val:
                best_val=val;best_epoch=epoch;since_best=0
                temp=best_ckpt.with_suffix('.tmp')
                torch.save({'model':model.state_dict(),'signature':signature,
                            'epoch':epoch,'mean':mean,'std':std,'val_forecast_l2':val},temp)
                os.replace(temp,best_ckpt)
            else:since_best+=1
        history.append(row)
        elapsed+=epoch_sec
        print(f"N={n:3} epoch={epoch:03}/{CFG['EPOCHS']} loss={row['train_normalized_mse']:.5e} "
              + (f"val={row['val_forecast_l2']:.5e} best={best_val:.5e}@{best_epoch}" if 'val_forecast_l2' in row else '')
              + f' sec={epoch_sec:.2f}',flush=True)
        if 'val_forecast_l2' in row:
            temp=latest_ckpt.with_suffix('.tmp')
            torch.save({
                'signature':signature,'epoch':epoch,'model':model.state_dict(),
                'optimizer':optimizer.state_dict(),'best_val':best_val,
                'best_epoch':best_epoch,'since_best':since_best,'elapsed':elapsed,
                'history':history,'loader_rng':gen.get_state(),
                'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
                'py_rng':random.getstate(),'numpy_rng':np.random.get_state(),
            },temp)
            os.replace(temp,latest_ckpt)
            if since_best>=CFG['PATIENCE_EVALS']:
                print('Early stop after',epoch,'epochs; selected',best_epoch)
                break

    assert best_ckpt.is_file(),'No best validation checkpoint exists.'
    best=torch.load(best_ckpt,map_location=DEVICE,weights_only=False)
    assert best['signature']==signature
    model.load_state_dict(best['model'])
    # Test is touched only now, after validation-only selection.
    torch.cuda.synchronize();t0=time.perf_counter()
    prediction=forecast(model,TEST,mean,std)
    torch.cuda.synchronize();test_inference_seconds=time.perf_counter()-t0
    result=metrics(prediction,TEST)
    result.update({'pde':CFG['PDE'],'n_train':n,'history_length':H,'k':K,
                   'n_val':len(VAL),'n_test':len(TEST),
                   'params':sum(p.numel() for p in model.parameters() if p.requires_grad),
                   'real_scalar_dof':sum(p.numel()*(2 if p.is_complex() else 1) for p in model.parameters() if p.requires_grad),
                   'train_seconds':round(elapsed,2),'test_inference_seconds':round(test_inference_seconds,2),
                   'best_epoch':best_epoch,'best_val_forecast_l2':best_val,
                   'epochs_completed':history[-1]['epoch'],'train_windows_per_trajectory':len(tr_ds.starts),
                   'input_frames':10,'no_teacher_forcing_at_test':True,
                   'full_error_scope':'forecast frames 10..100 (trapezoidal physical-time L2)',
                   'signature':signature,'source_commit':SOURCE_COMMIT,'source_sha256':source_hash,
                   'gpu':torch.cuda.get_device_name(0),'config':CFG})

    (out/'train_log.json').write_text(json.dumps(history,indent=2))
    plt.figure(figsize=(7,4))
    plt.plot(TIMES[H:],result['l2_vs_time_mean'])
    plt.xlabel('Physical time');plt.ylabel('Mean absolute spatial $L^2$ error')
    plt.title(f"{CFG['PDE']} AWFNO-H10-K10 N={n}")
    plt.tight_layout();plt.savefig(out/'l2_vs_time.png',dpi=160);plt.close()
    plt.figure(figsize=(7,4));plt.plot([r['epoch'] for r in history],
                     [r['train_normalized_mse'] for r in history],label='Training window MSE')
    plt.yscale('log');plt.xlabel('Epoch');plt.legend();plt.tight_layout()
    plt.savefig(out/'loss.png',dpi=160);plt.close()
    result_path.write_text(json.dumps(result,indent=2,allow_nan=False))
    print('TEST RESULT',n,'forecast L2',result['forecast_l2_mean'],
          'final L2',result['final_l2_mean'],'params',result['params'],flush=True)
    del model,prediction
    torch.cuda.empty_cache();gc.collect()
    return result


In [ ]:

# All training sizes; change CFG['N_SIZES'] above to [25] for an initial GPU smoke run.
results=[]
for n in CFG['N_SIZES']:
    row=train_one(n)
    results.append(row)
    fields=['pde','n_train','forecast_l2_mean','forecast_l2_sd','final_l2_mean','final_l2_sd',
            'first_forecast_l2_mean','params','real_scalar_dof','train_seconds',
            'test_inference_seconds','best_epoch','epochs_completed','input_frames','k',
            'n_val','n_test','signature']
    # Write incrementally: a disconnected Colab does not destroy preceding results.
    with (ROOT/'summary.csv').open('w',newline='') as fh:
        writer=csv.DictWriter(fh,fieldnames=fields,extrasaction='ignore')
        writer.writeheader();writer.writerows(results)
    (ROOT/'summary.json').write_text(json.dumps(results,indent=2,allow_nan=False))
print('Saved:',ROOT/'summary.csv')
print('Paste these into the H10 comparison table (forecast / final):')
for row in results:
    print(f"N={row['n_train']:3d}: {row['forecast_l2_mean']:.4f} / {row['final_l2_mean']:.4f}")


**Interpretation:** compare the two H10 models on forecast-only and final-time errors. Their input information is matched (ten exact frames), but training procedures are not identical. Do not mix H10 full errors with initial-condition-only `t0..t100` errors.